In [14]:
import pandas as pd

# Load dataset
df = pd.read_csv("df_with_pred_qualifying_v13.csv")

print(df.shape)

shape_2020_25 = df[df["season"] >= 2020].shape
df.head()

(2970, 45)


,season,round,DriverId,TeamId,grid_position,race_finishing_position,classified_position,status,raw_points,laps_completed,...,positive_qualifying_surprise,negative_qualifying_surprise,predicted_classification_points,predicted_classification_points_raw,race_gain,race_loss,recent_gain_exposure,recent_loss_exposure,exposure_history_count,net_recent_deviation
0,2020,16,aitken,williams,17,16,16,Finished,0.0,87,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
1,2019,1,albon,toro_rosso,13,14,14,+1 Lap,0.0,57,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,2019,2,albon,toro_rosso,12,9,9,Finished,2.0,57,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
3,2019,4,albon,toro_rosso,11,11,11,+1 Lap,0.0,50,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,2019,5,albon,toro_rosso,11,11,11,Finished,0.0,66,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [15]:
required_model_columns = [
    "qualifying_position",
    "driver_points_mean_last3",
    "driver_qualifying_mean_last3",
    "team_points_mean_last3",
    "team_qualifying_mean_last3",
    "driver_incident_rate_last10",
    "driver_previous_starts",
    "exposure_history_count",
    "positive_qualifying_surprise",
    "negative_qualifying_surprise",
    "recent_gain_exposure",
    "recent_loss_exposure",
    "qualifying_surprise",
    "net_recent_deviation",
    "Top10",
    "Podium",
    "IncidentDNF"
]

In [16]:
nan_counts = (
    df[df["season"] >= 2020][required_model_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(nan_counts)

positive_qualifying_surprise    25
qualifying_surprise             25
negative_qualifying_surprise    25
driver_qualifying_mean_last3    17
driver_incident_rate_last10     17
driver_points_mean_last3        17
team_points_mean_last3           8
team_qualifying_mean_last3       8
Podium                           0
Top10                            0
net_recent_deviation             0
qualifying_position              0
recent_loss_exposure             0
recent_gain_exposure             0
exposure_history_count           0
driver_previous_starts           0
IncidentDNF                      0
dtype: int64

In [20]:
df = (
    df[df["season"] >= 2020]
    .dropna(subset=required_model_columns)
    .copy()
    .reset_index(drop=True)
)
shape_2020_25_others = df.shape
df

,season,round,DriverId,TeamId,grid_position,race_finishing_position,classified_position,status,raw_points,laps_completed,...,positive_qualifying_surprise,negative_qualifying_surprise,predicted_classification_points,predicted_classification_points_raw,race_gain,race_loss,recent_gain_exposure,recent_loss_exposure,exposure_history_count,net_recent_deviation
0,2020,1,albon,red_bull,4,13,13,Electronics,0.0,67,...,1.0,0.0,12.0,9.386359,0.0,12.0,0.0,0.0,0.0,0.0
1,2020,2,albon,red_bull,6,4,4,Finished,12.0,71,...,0.0,2.0,2.0,6.229112,10.0,0.0,0.0,12.0,1.0,-12.0
2,2020,3,albon,red_bull,13,5,5,Finished,10.0,70,...,0.0,8.0,4.0,7.333508,6.0,0.0,10.0,12.0,2.0,-2.0
3,2020,4,albon,red_bull,12,8,8,Finished,4.0,52,...,0.0,5.0,8.0,5.820623,0.0,4.0,16.0,12.0,3.0,4.0
4,2020,5,albon,red_bull,9,5,5,Finished,10.0,52,...,3.0,0.0,12.0,7.859527,0.0,2.0,16.0,4.0,3.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2102,2024,17,zhou,sauber,17,14,14,Finished,0.0,51,...,2.0,0.0,0.0,0.463901,0.0,0.0,0.0,0.0,3.0,0.0
2103,2024,18,zhou,sauber,20,15,15,Lapped,0.0,61,...,0.0,-0.0,0.0,0.631984,0.0,0.0,0.0,0.0,3.0,0.0
2104,2024,20,zhou,sauber,19,15,15,Lapped,0.0,70,...,0.0,-0.0,0.0,0.358523,0.0,0.0,0.0,0.0,3.0,0.0
2105,2024,22,zhou,sauber,13,13,13,Finished,0.0,50,...,7.0,0.0,0.0,0.491869,0.0,0.0,0.0,0.0,3.0,0.0


In [21]:
shape_diff = shape_2020_25[0] - shape_2020_25_others[0]
print(shape_diff)

25


In [22]:
y1 = df["Top10"]        # Primary performance target
y2 = df["Podium"]       # Exceptional-performance target
y3 = df["IncidentDNF"]  # Risk-related target

control_features = [
    "driver_incident_rate_last10",
    "driver_previous_starts",
    "exposure_history_count"
]

baseline_performance_features = [
    "qualifying_position",
    "driver_points_mean_last3",
    "driver_qualifying_mean_last3",
    "team_points_mean_last3",
    "team_qualifying_mean_last3"
]

categorical_features = [
    "DriverId",
    "TeamId",
    "location",
    "season"
]

baseline_features = (
    baseline_performance_features
    + control_features
    + categorical_features
)



### Model 0: Conventional baseline

In [23]:
X0_features = baseline_features

X0 = df[X0_features]

### Model 1: Qualifying expectation

In [24]:
X1_features = baseline_features + [
    "positive_qualifying_surprise",
    "negative_qualifying_surprise"
]

X1 = df[X1_features]

### Model 2: Gain-loss exposure

In [25]:
X2_features = baseline_features + [
    "recent_gain_exposure",
    "recent_loss_exposure"
]

X2 = df[X2_features]

### Model 3: Full asymmetric model

In [26]:
X3_features = baseline_features + [
    "positive_qualifying_surprise",
    "negative_qualifying_surprise",
    "recent_gain_exposure",
    "recent_loss_exposure"
]

X3 = df[X3_features]

### Model 4: Symmetric comparison model

In [27]:
X4_features = baseline_features + [
    "qualifying_surprise",
    "net_recent_deviation"
]

X4 = df[X4_features]

### All models

In [28]:
feature_sets = {
    "M0_baseline": X0_features,
    "M1_qualifying_expectation": X1_features,
    "M2_gain_loss": X2_features,
    "M3_full_asymmetric": X3_features,
    "M4_symmetric": X4_features
}

targets = {
    "Top10": "Top10",
    "Podium": "Podium",
    "IncidentDNF": "IncidentDNF"
}

In [29]:
for model_name, features in feature_sets.items():
    print(f"\n{model_name}")
    print(features)


M0_baseline
['qualifying_position', 'driver_points_mean_last3', 'driver_qualifying_mean_last3', 'team_points_mean_last3', 'team_qualifying_mean_last3', 'driver_incident_rate_last10', 'driver_previous_starts', 'exposure_history_count', 'DriverId', 'TeamId', 'location', 'season']

M1_qualifying_expectation
['qualifying_position', 'driver_points_mean_last3', 'driver_qualifying_mean_last3', 'team_points_mean_last3', 'team_qualifying_mean_last3', 'driver_incident_rate_last10', 'driver_previous_starts', 'exposure_history_count', 'DriverId', 'TeamId', 'location', 'season', 'positive_qualifying_surprise', 'negative_qualifying_surprise']

M2_gain_loss
['qualifying_position', 'driver_points_mean_last3', 'driver_qualifying_mean_last3', 'team_points_mean_last3', 'team_qualifying_mean_last3', 'driver_incident_rate_last10', 'driver_previous_starts', 'exposure_history_count', 'DriverId', 'TeamId', 'location', 'season', 'recent_gain_exposure', 'recent_loss_exposure']

M3_full_asymmetric
['qualifying_